# 00 Foundations — Linear Algebra

**Status:** Complete

**Learning outcome:** Confident with matrix multiply, transpose, dot product as NumPy operations. Can predict output shapes before running code.

## Why This Matters

Every neural network is a sequence of matrix multiplications and element-wise nonlinearities. The most common bug in deep learning code is a shape mismatch — two matrices that cannot be multiplied because their inner dimensions disagree. If you can predict the output shape of every operation before running it, you can debug shape errors from the traceback alone.

This notebook drills matrix operations until shape reasoning is automatic. We implement matrix multiply from scratch, then verify against NumPy, and build geometric intuition for what linear transformations do to vectors.

## Theory Sketch

A **matrix** $A \in \mathbb{R}^{m \times n}$ has $m$ rows and $n$ columns. Its **shape** is $(m, n)$.

**Matrix multiply** $C = A \cdot B$ where $A$ is $(m, n)$ and $B$ is $(n, p)$ gives $C$ with shape $(m, p)$:

$$C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}$$

The inner dimensions must match: $A$’s columns = $B$’s rows.

**Transpose** $A^T$ swaps rows and columns: $(m, n) \to (n, m)$.

**Eigendecomposition** of a square matrix $A$: $Av = \lambda v$ where $v$ is an eigenvector and $\lambda$ is the corresponding eigenvalue. Geometrically, $A$ stretches $v$ by factor $\lambda$ without changing its direction.

## From-Scratch Implementation (NumPy)

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# --- 1. Vectors and matrices as arrays ---
v = np.array([1.0, 2.0, 3.0])           # vector: shape (3,)
M = np.array([[1, 2], [3, 4], [5, 6]])   # matrix: shape (3, 2)

print(f"Vector v: shape {v.shape}")
print(f"Matrix M: shape {M.shape}")
print(f"M transpose: shape {M.T.shape}")
print()

# --- 2. Matrix multiply from scratch ---
def matmul_scratch(A, B):
    """Matrix multiply using nested loops."""
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"Inner dimensions must match: {n} != {n2}"
    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C

A = np.random.randn(3, 4)
B = np.random.randn(4, 2)
C_scratch = matmul_scratch(A, B)
C_numpy = A @ B

print(f"A: {A.shape}, B: {B.shape} => C: {C_scratch.shape}")
print(f"Max difference (scratch vs np): {np.max(np.abs(C_scratch - C_numpy)):.2e}")
assert np.allclose(C_scratch, C_numpy), "Scratch matmul does not match NumPy!"
print("Scratch matmul matches NumPy exactly.")

Vector v: shape (3,)
Matrix M: shape (3, 2)
M transpose: shape (2, 3)

A: (3, 4), B: (4, 2) => C: (3, 2)
Max difference (scratch vs np): 1.67e-16
Scratch matmul matches NumPy exactly.


In [2]:
# --- 3. Shape rules: (m,n) @ (n,p) -> (m,p) --- drill with 5 examples ---
shape_examples = [
    ((2, 3), (3, 5)),   # => (2, 5)
    ((1, 4), (4, 1)),   # => (1, 1)  -- dot product as matmul
    ((5, 2), (2, 3)),   # => (5, 3)
    ((3, 3), (3, 3)),   # => (3, 3)  -- square matrices
    ((10, 1), (1, 7)),  # => (10, 7) -- outer-product-like
]

print(f"{'A shape':>10} {'B shape':>10} {'Expected':>10} {'Actual':>10} {'Match':>6}")
print("-" * 50)
for sa, sb in shape_examples:
    expected = (sa[0], sb[1])
    A_test = np.random.randn(*sa)
    B_test = np.random.randn(*sb)
    actual = (A_test @ B_test).shape
    match = actual == expected
    print(f"{str(sa):>10} {str(sb):>10} {str(expected):>10} {str(actual):>10} {str(match):>6}")
    assert match, f"Shape mismatch: expected {expected}, got {actual}"

print("\nAll 5 shape predictions correct.")

   A shape    B shape   Expected     Actual  Match
--------------------------------------------------
    (2, 3)     (3, 5)     (2, 5)     (2, 5)   True
    (1, 4)     (4, 1)     (1, 1)     (1, 1)   True
    (5, 2)     (2, 3)     (5, 3)     (5, 3)   True
    (3, 3)     (3, 3)     (3, 3)     (3, 3)   True
   (10, 1)     (1, 7)    (10, 7)    (10, 7)   True

All 5 shape predictions correct.


---
**Understanding Matrix Multiplication and Shape Rules**

**Plain language:** Matrix multiplication is like a recipe factory. Matrix A describes "how much of each ingredient to use" and matrix B describes "how much of each ingredient is available." The result tells you "what you can make." The key rule is simple: the number of columns in A must equal the number of rows in B — like making sure the plug fits the socket. If A is a 3×4 table and B is a 4×2 table, the "4" in the middle must match, and your answer is a 3×2 table.

**Building intuition:** Each entry $C_{ij}$ of the result is a dot product — the i-th row of A dotted with the j-th column of B. This is why the inner dimensions must agree: you're pairing elements one-to-one. The shape rule $(m, n) \times (n, p) \to (m, p)$ is the single most important formula for debugging neural networks — if your code crashes with "matmul: input operand has a mismatch", you need to check these three numbers. In neural networks, this rule determines how information flows: a weight matrix $W$ of shape (input_dim, output_dim) transforms activations from one size to another.

**Formal statement:** Given $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{n \times p}$, the product $C = AB \in \mathbb{R}^{m \times p}$ is defined by $C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}$. Computational complexity is $\mathcal{O}(mnp)$. The operation is associative $(AB)C = A(BC)$ and distributive $A(B+C) = AB + AC$, but **not** commutative: $AB \neq BA$ in general.

---

In [ ]:
# --- Visual: Shape Rule Diagram ---
fig, ax = plt.subplots(figsize=(10, 3))
ax.set_xlim(0, 10); ax.set_ylim(0, 3)
ax.set_aspect('equal'); ax.axis('off')

# Matrix A rectangle
a_rect = plt.Rectangle((0.5, 0.5), 1.5, 2.0, fill=True, facecolor='#4ECDC4', alpha=0.5, edgecolor='black', lw=2)
ax.add_patch(a_rect)
ax.text(1.25, 1.5, 'A', fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(1.25, 0.15, 'n cols', fontsize=10, ha='center', color='#E74C3C', fontweight='bold')
ax.text(0.15, 1.5, 'm\nrows', fontsize=10, ha='center', va='center')

# Matrix B rectangle
b_rect = plt.Rectangle((3.0, 0.8), 2.0, 1.5, fill=True, facecolor='#45B7D1', alpha=0.5, edgecolor='black', lw=2)
ax.add_patch(b_rect)
ax.text(4.0, 1.55, 'B', fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(4.0, 0.45, 'p cols', fontsize=10, ha='center')
ax.text(2.65, 1.55, 'n\nrows', fontsize=10, ha='center', va='center', color='#E74C3C', fontweight='bold')

# "Must match" bracket
ax.annotate('', xy=(1.25, 2.7), xytext=(4.0, 2.7),
            arrowprops=dict(arrowstyle='<->', color='#E74C3C', lw=2))
ax.text(2.625, 2.85, 'must match (n)', fontsize=11, ha='center', color='#E74C3C', fontweight='bold')

# Equals sign
ax.text(5.5, 1.5, '=', fontsize=24, ha='center', va='center')

# Result C rectangle
c_rect = plt.Rectangle((6.2, 0.5), 2.0, 2.0, fill=True, facecolor='#96CEB4', alpha=0.5, edgecolor='black', lw=2)
ax.add_patch(c_rect)
ax.text(7.2, 1.5, 'C', fontsize=16, ha='center', va='center', fontweight='bold')
ax.text(7.2, 0.15, 'p cols', fontsize=10, ha='center')
ax.text(5.85, 1.5, 'm\nrows', fontsize=10, ha='center', va='center')

ax.set_title('Matrix Multiply Shape Rule: (m, n) @ (n, p) → (m, p)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/shape_rule_diagram.png', dpi=100, bbox_inches='tight')
plt.show()
print("Shape rule diagram saved.")

In [3]:
# --- 4. Transpose, inverse, determinant ---
S = np.array([[4.0, 2.0],
              [1.0, 3.0]])

print(f"S =\n{S}")
print(f"\nS^T =\n{S.T}")
print(f"\ndet(S) = {np.linalg.det(S):.4f}")

S_inv = np.linalg.inv(S)
print(f"\nS^(-1) =\n{S_inv}")

# Verify: S @ S^(-1) = I
identity_check = S @ S_inv
print(f"\nS @ S^(-1) =\n{identity_check}")
assert np.allclose(identity_check, np.eye(2)), "Inverse verification failed!"
print("Inverse verified: S @ S^(-1) = I")

S =
[[4. 2.]
 [1. 3.]]

S^T =
[[4. 1.]
 [2. 3.]]

det(S) = 10.0000

S^(-1) =
[[ 0.3 -0.2]
 [-0.1  0.4]]

S @ S^(-1) =
[[ 1.00000000e+00  0.00000000e+00]
 [-2.77555756e-17  1.00000000e+00]]
Inverse verified: S @ S^(-1) = I


---
**Understanding Transpose, Inverse, and Determinant**

**Plain language:** Transpose is like flipping a spreadsheet — rows become columns and columns become rows. The inverse of a matrix is like an "undo button" — if a matrix transforms your data, its inverse transforms it back to the original. The determinant tells you how much the matrix scales the area (or volume) of shapes: a determinant of 2 means areas double, a determinant of 0 means the matrix squashes everything flat.

**Building intuition:** The transpose $A^T$ swaps the (i,j) entry with the (j,i) entry. In neural networks, transposes appear everywhere in backpropagation: if the forward pass multiplies by $W$, the backward pass multiplies by $W^T$. The inverse $A^{-1}$ exists only when $\det(A) \neq 0$ — meaning the transformation doesn't collapse any dimension. A zero determinant signals a "lossy" transformation that cannot be reversed.

**Formal statement:** For $A \in \mathbb{R}^{m \times n}$, $(A^T)_{ij} = A_{ji}$, giving shape $(n, m)$. Key identity: $(AB)^T = B^T A^T$. For square $A \in \mathbb{R}^{n \times n}$, the inverse satisfies $AA^{-1} = A^{-1}A = I_n$ and exists iff $\det(A) \neq 0$. The determinant gives the signed volume scaling factor of the linear map: $\det(A) = \prod_i \lambda_i$ where $\lambda_i$ are eigenvalues.

---

In [4]:
# --- 5. Eigenvalues --- one worked example (2x2) ---
# Symmetric matrix for real eigenvalues
E = np.array([[2.0, 1.0],
              [1.0, 3.0]])

eigenvalues, eigenvectors = np.linalg.eig(E)
print(f"Matrix E =\n{E}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

# Verify: E @ v = lambda * v for each eigenpair
for i in range(2):
    lam = eigenvalues[i]
    v = eigenvectors[:, i]
    lhs = E @ v
    rhs = lam * v
    print(f"\nEigenpair {i}: lambda={lam:.4f}")
    print(f"  E @ v   = {lhs}")
    print(f"  lam * v = {rhs}")
    assert np.allclose(lhs, rhs), f"Eigenpair {i} verification failed!"

print("\nBoth eigenpairs verified: E @ v = lambda * v")
print(f"\nGeometric meaning: E stretches along eigenvector directions by factors {eigenvalues[0]:.2f} and {eigenvalues[1]:.2f}")

Matrix E =
[[2. 1.]
 [1. 3.]]

Eigenvalues: [1.38196601 3.61803399]
Eigenvectors (columns):
[[-0.85065081 -0.52573111]
 [ 0.52573111 -0.85065081]]

Eigenpair 0: lambda=1.3820
  E @ v   = [-1.1755705   0.72654253]
  lam * v = [-1.1755705   0.72654253]

Eigenpair 1: lambda=3.6180
  E @ v   = [-1.90211303 -3.07768354]
  lam * v = [-1.90211303 -3.07768354]

Both eigenpairs verified: E @ v = lambda * v

Geometric meaning: E stretches along eigenvector directions by factors 1.38 and 3.62


In [ ]:
# --- Visual: Eigenvector Stretching ---
# Show how matrix E transforms its eigenvectors vs arbitrary vectors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta = np.linspace(0, 2*np.pi, 100)
circle = np.stack([np.cos(theta), np.sin(theta)])
E_transformed = E @ circle

for ax in axes:
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)

# Panel 1: Before — unit circle with eigenvectors highlighted
axes[0].plot(circle[0], circle[1], 'b-', alpha=0.3, lw=1)
colors = ['#E74C3C', '#2ECC71']
for i in range(2):
    ev = eigenvectors[:, i]
    axes[0].annotate('', xy=ev*1.5, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=colors[i], lw=3))
    axes[0].text(ev[0]*1.7, ev[1]*1.7,
                 f'v{i+1} (λ={eigenvalues[i]:.2f})',
                 color=colors[i], fontsize=11, fontweight='bold')
axes[0].set_title('Before: eigenvectors on unit circle', fontsize=12)

# Panel 2: After — transformed, eigenvectors only stretched
axes[1].plot(E_transformed[0], E_transformed[1], 'b-', alpha=0.3, lw=1)
for i in range(2):
    ev = eigenvectors[:, i]
    ev_transformed = E @ (ev * 1.5)  # = lambda * ev * 1.5
    axes[1].annotate('', xy=ev_transformed, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=colors[i], lw=3))
    axes[1].text(ev_transformed[0]*1.15, ev_transformed[1]*1.15,
                 f'E·v{i+1} = {eigenvalues[i]:.2f}·v{i+1}',
                 color=colors[i], fontsize=10, fontweight='bold')
axes[1].set_title('After: eigenvectors stretched, not rotated', fontsize=12)

plt.suptitle('Eigenvalue Geometry: E stretches along eigenvector directions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/eigenvector_stretching.png', dpi=100, bbox_inches='tight')
plt.show()
print("Eigenvector stretching visualisation saved.")

---
**Understanding Eigenvalues and Eigenvectors**

**Plain language:** Imagine pushing on a door. Most pushes make it both swing *and* slide. But there are special directions where your push only makes the door move more (or less) in the *same* direction — no sideways drift. Those special directions are eigenvectors, and how much the door moves is the eigenvalue. A matrix has its own "natural directions" that it simply stretches or shrinks without rotating.

**Building intuition:** An eigenvector $v$ of matrix $A$ is a direction that $A$ does not rotate — it only scales it by a factor $\lambda$ (the eigenvalue). If $\lambda > 1$, the vector stretches; if $0 < \lambda < 1$, it shrinks; if $\lambda < 0$, it flips direction. For symmetric matrices (common in ML — think covariance matrices, Hessians), all eigenvalues are real and eigenvectors are orthogonal, giving a clean decomposition of the transformation into independent axes of stretching.

**Formal statement:** For a square matrix $A \in \mathbb{R}^{n \times n}$, scalar $\lambda$ and nonzero vector $v$ satisfying $Av = \lambda v$ are an eigenpair. The eigenvalues are roots of the characteristic polynomial $\det(A - \lambda I) = 0$. For symmetric $A = A^T$, the spectral theorem guarantees $A = Q \Lambda Q^T$ where $Q$ is orthogonal and $\Lambda = \text{diag}(\lambda_1, \dots, \lambda_n)$.

---

In [5]:
# --- 6. Shape prediction exercise: 10 operations ---
print("Shape Prediction Exercise")
print("========================")
print("Predict the output shape, then check.\n")

exercises = [
    ("A(3,4) @ B(4,2)",        lambda: np.random.randn(3,4) @ np.random.randn(4,2),        (3,2)),
    ("A(5,1) @ B(1,5)",        lambda: np.random.randn(5,1) @ np.random.randn(1,5),        (5,5)),
    ("A(2,3) @ B(3,1)",        lambda: np.random.randn(2,3) @ np.random.randn(3,1),        (2,1)),
    ("A(1,10) @ B(10,1)",      lambda: np.random.randn(1,10) @ np.random.randn(10,1),      (1,1)),
    ("A(4,4) @ A(4,4)",        lambda: (lambda a: a @ a)(np.random.randn(4,4)),             (4,4)),
    ("A(3,2).T @ B(3,5)",      lambda: np.random.randn(3,2).T @ np.random.randn(3,5),      (2,5)),
    ("A(6,3) @ B(3,3) @ C(3,2)", lambda: np.random.randn(6,3) @ np.random.randn(3,3) @ np.random.randn(3,2), (6,2)),
    ("(A(2,5) @ B(5,3)).T",    lambda: (np.random.randn(2,5) @ np.random.randn(5,3)).T,    (3,2)),
    ("A(8,1) * B(8,1) elem",   lambda: np.random.randn(8,1) * np.random.randn(8,1),        (8,1)),
    ("A(3,4) + B(3,4) elem",   lambda: np.random.randn(3,4) + np.random.randn(3,4),        (3,4)),
]

print(f"{'#':>2} {'Operation':<30} {'Predicted':>10} {'Actual':>10} {'OK':>4}")
print("-" * 60)
for i, (desc, op, predicted) in enumerate(exercises, 1):
    result = op()
    actual = result.shape
    ok = actual == predicted
    print(f"{i:2d} {desc:<30} {str(predicted):>10} {str(actual):>10} {'  Y' if ok else '  N'}")
    assert ok, f"Exercise {i}: predicted {predicted}, got {actual}"

print("\nAll 10 shape predictions correct.")

Shape Prediction Exercise
Predict the output shape, then check.

 # Operation                       Predicted     Actual   OK
------------------------------------------------------------
 1 A(3,4) @ B(4,2)                    (3, 2)     (3, 2)   Y
 2 A(5,1) @ B(1,5)                    (5, 5)     (5, 5)   Y
 3 A(2,3) @ B(3,1)                    (2, 1)     (2, 1)   Y
 4 A(1,10) @ B(10,1)                  (1, 1)     (1, 1)   Y
 5 A(4,4) @ A(4,4)                    (4, 4)     (4, 4)   Y
 6 A(3,2).T @ B(3,5)                  (2, 5)     (2, 5)   Y
 7 A(6,3) @ B(3,3) @ C(3,2)           (6, 2)     (6, 2)   Y
 8 (A(2,5) @ B(5,3)).T                (3, 2)     (3, 2)   Y
 9 A(8,1) * B(8,1) elem               (8, 1)     (8, 1)   Y
10 A(3,4) + B(3,4) elem               (3, 4)     (3, 4)   Y

All 10 shape predictions correct.


## PyTorch Rewrite

In [6]:
import torch

# Same matrix multiply in PyTorch
A_t = torch.tensor(A, dtype=torch.float64)
B_t = torch.tensor(B, dtype=torch.float64)
C_t = A_t @ B_t

print(f"PyTorch matmul: {A_t.shape} @ {B_t.shape} = {C_t.shape}")
assert np.allclose(C_t.numpy(), C_numpy), "PyTorch result doesn't match NumPy!"
print("PyTorch matmul matches NumPy.")

# Eigendecomposition in PyTorch
E_t = torch.tensor(E, dtype=torch.float64)
eigvals_t, eigvecs_t = torch.linalg.eig(E_t)
print(f"\nPyTorch eigenvalues: {eigvals_t.real.numpy()}")
assert np.allclose(sorted(eigvals_t.real.numpy()), sorted(eigenvalues)), "Eigenvalue mismatch!"
print("PyTorch eigenvalues match NumPy.")

PyTorch matmul: torch.Size([3, 4]) @ torch.Size([4, 2]) = torch.Size([3, 2])
PyTorch matmul matches NumPy.

PyTorch eigenvalues: [1.38196601 3.61803399]
PyTorch eigenvalues match NumPy.


## Training Run

Not applicable for this foundations notebook — no model to train.

## Internal Probing

Visualise a 2D linear transformation: how a matrix transforms a set of vectors. We apply the matrix to unit circle vectors and see the resulting ellipse.

In [7]:
# 2D linear transformation visualisation
T = np.array([[2.0, 1.0],
              [0.5, 1.5]])

# Unit circle points
theta = np.linspace(0, 2*np.pi, 100)
circle = np.stack([np.cos(theta), np.sin(theta)])  # (2, 100)
transformed = T @ circle  # (2, 100)

# Basis vectors and their transforms
e1 = np.array([1, 0])
e2 = np.array([0, 1])
Te1 = T @ e1
Te2 = T @ e2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before transformation
axes[0].plot(circle[0], circle[1], 'b-', linewidth=1.5, label='Unit circle')
axes[0].annotate('', xy=e1, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
axes[0].annotate('', xy=e2, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='green', lw=2))
axes[0].text(e1[0]+0.1, e1[1]+0.1, 'e1', color='red', fontsize=12)
axes[0].text(e2[0]+0.1, e2[1]+0.1, 'e2', color='green', fontsize=12)
axes[0].set_xlim(-3, 3); axes[0].set_ylim(-3, 3)
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].set_title('Before: unit circle + basis vectors')

# After transformation
axes[1].plot(transformed[0], transformed[1], 'b-', linewidth=1.5, label='Transformed')
axes[1].annotate('', xy=Te1, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
axes[1].annotate('', xy=Te2, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='green', lw=2))
axes[1].text(Te1[0]+0.1, Te1[1]+0.1, f'T·e1={Te1}', color='red', fontsize=10)
axes[1].text(Te2[0]+0.1, Te2[1]+0.1, f'T·e2={Te2}', color='green', fontsize=10)
axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3)
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
axes[1].set_title(f'After: T = {T.tolist()}')

plt.suptitle('2D Linear Transformation: Rotation + Stretching', fontsize=14)
plt.tight_layout()
plt.show()
print("Transformation visualisation complete.")

Transformation visualisation complete.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_41239/4189433960.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Structured Interpretation

### Results
- From-scratch matrix multiply matches NumPy to machine precision
- All 5 shape-rule drills and 10 shape-prediction exercises correct
- Eigenpair verification: $Ev = \lambda v$ holds for both eigenpairs of the 2×2 test matrix
- PyTorch matmul and eigendecomposition match NumPy results

### Findings
- Shape is the primary debugging tool: if you know $(m,n) @ (n,p) \to (m,p)$, most matmul errors are diagnosable from the traceback
- Transpose as shape swap $(m,n) \to (n,m)$ is essential for computing gradients (Section 01)
- Eigenvalues describe how a matrix stretches space — visible in the transformation plot as the ellipse axes

### Implications
- Weight matrices in neural networks are linear transformations; each layer transforms the activation shape
- Backprop gradient formulas use transposes: $\frac{\partial L}{\partial W} = X^T \cdot \frac{\partial L}{\partial Z}$ (notebook 01/03)
- Eigenvalue analysis is used for understanding loss landscape curvature (Hessian eigenvalues)

### Considerations
- From-scratch matmul is $O(mnp)$ — NumPy’s BLAS backend is orders of magnitude faster
- `np.linalg.inv` is numerically unstable for ill-conditioned matrices; prefer `np.linalg.solve` when possible
- Broadcasting in NumPy/PyTorch adds implicit shape rules beyond matmul — learn these separately